# Pricing across asset classes

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb` (the standard pricing pipeline: tagged instrument JSON, `MarketContext` as JSON, valuation date, model key, and `ValuationResult.from_json`). This notebook reuses that same pipeline while swapping the **instrument** and, when needed, the **model** and **metrics**.

**Goal:** see one consistent workflow for rates, credit, equity, and FX—without leaving the JSON-oriented API.

## Concepts: a quick tour of asset classes

Finstack Quant groups instruments by **risk factor** and **cashflow shape**, not by desk org chart. The same pricing entry points apply everywhere:

- **Rates (money market & swaps):** PV from discounting (and forward projection for float legs). Curves: OIS discount, IBOR/RFR **forward** where needed.
- **Credit (CDS):** protection vs premium legs; survival from **hazard** curves. Model key is typically `hazard_rate` rather than pure `discounting`.
- **Equity options:** Black–Scholes–style models (`black76`) with **spot**, **discount**, **dividend yield**, and an **implied vol surface**.
- **FX options:** Garman–Kohlhagen-style stack: **two discount curves** (domestic/foreign), **FX spot** in the matrix, and a **vol surface** keyed by expiry × strike.

The pattern in each section below is deliberate and repeatable:

1. Build tagged instrument JSON (`{"type": "…", "spec": {…}}`).
2. Build a **fresh** typed `MarketContext` and serialize it with `to_json()`.
3. Call `price_instrument` or `price_instrument_with_metrics` with the right **model** string.
4. Parse with `ValuationResult.from_json` and read `price`, `currency`, and optional `get_metric`.

In [ ]:
import json
from datetime import date

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxMatrix,
    HazardCurve,
    MarketContext,
    VolSurface,
)
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import (
    list_standard_metrics_grouped,
    price_instrument,
    price_instrument_with_metrics,
    validate_instrument_json,
)

grouped = list_standard_metrics_grouped()
total = sum(len(v) for v in grouped.values())
print(f"Standard metrics ({total}) across {len(grouped)} groups:")
for group, metrics in grouped.items():
    print(f"  {group}: {', '.join(metrics[:5])}{'...' if len(metrics) > 5 else ''}")

## Shared market data (self-contained)

We construct curves on **`as_of = 2025-01-15`**: USD OIS discount, USD SOFR 3M forward, a single-name hazard curve for CDS, and EUR OIS for FX option domestic/foreign discounting.

`FxMatrix`, `VolSurface`, and scalar prices are inserted through the typed market API, so the serialized snapshot is ready for every pricer without JSON patching.

In [ ]:
AS_OF_DATE = date(2025, 1, 15)
AS_OF = AS_OF_DATE.isoformat()


def build_market_context_json(as_of: date) -> str:
    """Build a JSON market snapshot suitable for `price_instrument*`."""
    ois_usd = DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
        day_count="act_365f",
    )
    ois_eur = DiscountCurve(
        "EUR-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.988), (1.0, 0.975), (3.0, 0.92), (5.0, 0.85), (10.0, 0.70)],
        day_count="act_365f",
    )
    sofr_3m = ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        knots=[(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
        base_date=as_of,
        day_count="act_360",
    )
    hazard = HazardCurve(
        "CORP-HAZARD",
        as_of,
        [(0.5, 0.018), (1.0, 0.020), (3.0, 0.024), (5.0, 0.028), (10.0, 0.032)],
        recovery_rate=0.40,
    )
    fx = FxMatrix()
    fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)

    mc = MarketContext()
    mc.insert(ois_usd).insert(ois_eur).insert(sofr_3m).insert(hazard).insert_fx(fx)

    exp_eq = [0.25, 0.5, 1.0, 2.0]
    strikes_eq = [4000.0, 4300.0, 4500.0, 4700.0, 5000.0]
    exp_fx = [0.25, 0.5, 1.0, 2.0]
    strikes_fx = [0.9, 1.0, 1.1, 1.2, 1.3]
    mc.insert(
        VolSurface(
            "EQUITY-VOL",
            exp_eq,
            strikes_eq,
            [0.22] * (len(exp_eq) * len(strikes_eq)),
        )
    )
    mc.insert(
        VolSurface(
            "EURUSD-VOL",
            exp_fx,
            strikes_fx,
            [0.12] * (len(exp_fx) * len(strikes_fx)),
        )
    )
    mc.insert_price("EQUITY-SPOT", 4500.0, "USD")
    mc.insert_price("EQUITY-DIVYIELD", 0.015)

    return mc.to_json()


MARKET_JSON = build_market_context_json(AS_OF_DATE)
ROWS: list[dict] = []

print("as_of:", AS_OF)
print("market JSON length (chars):", len(MARKET_JSON))
print("surfaces:", len(json.loads(MARKET_JSON)["surfaces"]))

## 1. Deposit (rates / money market)

A **deposit** is the simplest discounting exercise: fixed coupon on a known day-count between `start_date` and `maturity`. Model: **`discounting`**.

We optionally run `validate_instrument_json` to pretty-print the canonical wire format.

In [3]:
deposit = json.dumps(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-1",
            "notional": {"amount": 1_000_000.0, "currency": "USD"},
            "start_date": "2025-01-15",
            "maturity": "2025-06-15",
            "day_count": "Act360",
            "quote_rate": 0.05,
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)

canon = validate_instrument_json(deposit)
print("canonical instrument JSON (first 220 chars):\n", canon[:220], "…")

out = price_instrument_with_metrics(
    deposit, MARKET_JSON, AS_OF, model="discounting", metrics=["dv01"]
)
vr = ValuationResult.from_json(out)
print(vr)
print("dv01:", vr.get_metric("dv01"))

ROWS.append(
    {
        "class": "Rates",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "dv01",
        "metric_value": vr.get_metric("dv01"),
    }
)

canonical instrument JSON (first 220 chars):
 {
  "spec": {
    "attributes": {},
    "bdc": "modified_following",
    "calendar_id": null,
    "day_count": "Act360",
    "discount_curve_id": "USD-OIS",
    "id": "DEP-1",
    "maturity": "2025-06-15",
    "notional" …
ValuationResult(id="DEP-1", price=1008301.1763, currency=USD, metrics=1)
dv01: -41.71328155341325


## 2. Interest rate swap (IRS)

Plain **fixed-for-floating** swap: fixed leg vs SOFR-linked float with `forward_curve_id`. Still **`discounting`** for the standard registry path.

**Seasoning:** if the floating leg has resets **before** `as_of`, the engine may require historical fixings. Here we use an **unseasoned** swap starting `2025-04-15`.

**Fixed rate:** The fixed coupon is set near **par (~4.5%)** for this OIS/SOFR snapshot so the swap PV is not dominated by a large off-market fixed leg (a **3%** coupon would be deeply off-market vs forwards and is fine for demos only).

In [ ]:
irs = json.dumps(
    {
        "type": "interest_rate_swap",
        "spec": {
            "id": "IRS-5Y-USD",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "fixed": {
                "discount_curve_id": "USD-OIS",
                "rate": 0.045,
                "frequency": {"count": 6, "unit": "months"},
                "day_count": "Thirty360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "start": "2025-04-15",
                "end": "2030-04-15",
                "par_method": None,
                "compounding_simple": True,
            },
            "float": {
                "discount_curve_id": "USD-OIS",
                "forward_curve_id": "USD-SOFR-3M",
                "spread_bp": 0.0,
                "frequency": {"count": 3, "unit": "months"},
                "day_count": "Act360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "reset_lag_days": 2,
                "start": "2025-04-15",
                "end": "2030-04-15",
                "compounding": "Simple",
            },
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    irs, MARKET_JSON, AS_OF, model="discounting", metrics=["dv01"]
)
vr = ValuationResult.from_json(out)
print(vr)
print("dv01:", vr.get_metric("dv01"))
ROWS.append(
    {
        "class": "Rates",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "dv01",
        "metric_value": vr.get_metric("dv01"),
    }
)


## 3. Credit default swap (CDS)

Single-name **CDS**: premium leg vs protection leg tied to `credit_curve_id` (`CORP-HAZARD` here). Use model **`hazard_rate`**.

JSON **`convention`** uses snake_case (e.g. `isda_na`), not `IsdaNa`.

In [ ]:
cds = json.dumps(
    {
        "type": "credit_default_swap",
        "spec": {
            "id": "CDS-CORP-5Y",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "convention": "isda_na",
            "premium": {
                "start": "2025-03-20",
                "end": "2030-03-20",
                "frequency": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "bdc": "following",
                "calendar_id": None,
                "day_count": "Act360",
                "spread_bp": 100.0,
                "discount_curve_id": "USD-OIS",
            },
            "protection": {
                "credit_curve_id": "CORP-HAZARD",
                "recovery_rate": 0.4,
                "settlement_delay": 3,
            },
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    cds, MARKET_JSON, AS_OF, model="hazard_rate", metrics=["cs01"]
)
vr = ValuationResult.from_json(out)
print(vr)
print("cs01:", vr.get_metric("cs01"))
ROWS.append(
    {
        "class": "Credit",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "cs01",
        "metric_value": vr.get_metric("cs01"),
    }
)


## 4. Equity option

Vanilla **equity** call: `black76` model, `spot_id` / `vol_surface_id` / optional `div_yield_id` resolved from the market snapshot (`EQUITY-SPOT`, `EQUITY-VOL`, `EQUITY-DIVYIELD`).

The wire schema uses a **numeric** `strike` and `notional` as `Money` (not `contract_size`).

In [ ]:
equity_option = json.dumps(
    {
        "type": "equity_option",
        "spec": {
            "id": "SPX-CALL-4500",
            "underlying_ticker": "SPX",
            "strike": 4500.0,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "notional": {"amount": 100.0, "currency": "USD"},
            "day_count": "Act365F",
            "settlement": "cash",
            "discount_curve_id": "USD-OIS",
            "spot_id": "EQUITY-SPOT",
            "vol_surface_id": "EQUITY-VOL",
            "div_yield_id": "EQUITY-DIVYIELD",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    equity_option,
    MARKET_JSON,
    AS_OF,
    model="black76",
    metrics=["delta"],
)
vr = ValuationResult.from_json(out)
print(vr)
print("delta:", vr.get_metric("delta"))
ROWS.append(
    {
        "class": "Equity",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "delta",
        "metric_value": vr.get_metric("delta"),
    }
)


## 5. FX option

EUR/USD **call**: domestic discount `USD-OIS`, foreign `EUR-OIS`, `vol_surface_id` `EURUSD-VOL`, and FX spot from the typed matrix.

Model: **`black76`**. We request **`delta`** here (registered for this pricer); it is expressed in the option’s quoting currency and scales with notional.

In [ ]:
fx_option = json.dumps(
    {
        "type": "fx_option",
        "spec": {
            "id": "FXOPT-EURUSD-CALL",
            "base_currency": "EUR",
            "quote_currency": "USD",
            "strike": 1.12,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "day_count": "Act365F",
            "notional": {"amount": 1_000_000.0, "currency": "EUR"},
            "settlement": "cash",
            "domestic_discount_curve_id": "USD-OIS",
            "foreign_discount_curve_id": "EUR-OIS",
            "vol_surface_id": "EURUSD-VOL",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

out = price_instrument_with_metrics(
    fx_option,
    MARKET_JSON,
    AS_OF,
    model="black76",
    metrics=["delta"],
)
vr = ValuationResult.from_json(out)
print(vr)
print("delta:", vr.get_metric("delta"))
ROWS.append(
    {
        "class": "FX",
        "instrument_id": vr.instrument_id,
        "price": vr.price,
        "ccy": vr.currency,
        "key_metric": "delta",
        "metric_value": vr.get_metric("delta"),
    }
)


## Mini-example: side-by-side results

The table below collects **instrument id**, **PV**, **currency**, and one **requested metric** per row. Metrics are not uniform across asset classes (rates `dv01`, credit `cs01`, options `delta`)—that is intentional: it mirrors how desks ask different risk questions.

In [8]:
hdr = f"{'class':<8} {'instrument_id':<16} {'price':>14} {'ccy':<4} {'metric':<10} {'value':>12}"
print(hdr)
print("-" * len(hdr))
for row in ROWS:
    p = row["price"]
    p_str = f"{p:,.4f}" if p is not None else "n/a"
    m = row["metric_value"]
    m_str = f"{m:,.6f}" if m is not None else "n/a"
    print(
        f"{row['class']:<8} {row['instrument_id']:<16} {p_str:>14} "
        f"{row['ccy']:<4} {row['key_metric']:<10} {m_str:>12}"
    )
print("rows:", len(ROWS))

class    instrument_id             price ccy  metric            value
---------------------------------------------------------------------
Rates    DEP-1            1,008,301.1763 USD  dv01         -41.713282
Rates    IRS-5Y-USD          78,723.9670 USD  dv01       4,463.497801
Credit   CDS-CORP-5Y        205,053.1150 USD  cs01       2,508.442673
Equity   SPX-CALL-4500       50,511.4209 USD  delta         57.430250
FX       FXOPT-EURUSD-CALL    55,336.6037 USD  delta      481,986.125731
rows: 5


## Commodity (forward)

Commodity instruments use `PriceCurve` (or `ForwardCurve`) for the underlying + a discount curve. Both JSON and typed paths are supported.


In [ ]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/pricing_across_asset_classes.json").read_text())

import json
from datetime import date
from finstack_quant.core.market_data import DiscountCurve, MarketContext, PriceCurve
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json

commodity_fwd = _NOTEBOOK_DATA['commodity_fwd']
cf_json = json.dumps(commodity_fwd)
print("validate:", validate_instrument_json(cf_json)[:120], "…")

# Market: discount + price curve backing the forward id
asof_d = date(2025, 1, 15)
pc = PriceCurve("WTI-FORWARD", asof_d, [(0.0, 70.0), (1.0, 71.0)], day_count="act_365f")
mkt = MarketContext()
mkt.insert(DiscountCurve("USD-OIS", asof_d, [(0.0,1.0),(1.0,0.95)], day_count="act_365f"))
mkt.insert(pc)

out = price_instrument_with_metrics(cf_json, mkt, AS_OF)
vr = ValuationResult.from_json(out)
print("JSON path PV:", round(vr.price, 2))
print("Canonical JSON path complete.")


## Takeaways

- **One pipeline:** `validate_instrument_json` → `price_instrument*` → `ValuationResult.from_json` works across rates, credit, equity, and FX; `02_pricing/pricing_fundamentals.ipynb` is the reference for that skeleton.
- **Model keys matter:** `discounting` for deposits and swaps, `hazard_rate` for CDS, `black76` for vanilla equity and FX options in the default registry wiring.
- **Market data is explicit:** curves must cover every `*_curve_id`; options need typed **surfaces** and **scalar prices**; FX options need the relevant pair in `FxMatrix`.
- **Deep dives:** follow the curriculum notebooks on **interest_rate_swap** / **credit_default_swap** / **equity_option** / **fx_option** instrument JSON (and the foundations notebook on **curves and `MarketContext`**) for fuller schedules, conventions, and metric sets.

**Cross-references (repo):** instrument schemas under `finstack-quant/valuations/schemas/instruments/`, worked JSON under `finstack-quant/valuations/tests/instruments/json_examples/`, and the market snapshot format in `finstack-quant/core` (`MarketContext` serde / `VolSurface` grid layout: `vols_row_major`).